# Analysis of pp Events: Event Activity Observables

Firstly, we load libraries.

In [ ]:
import argparse
import os
import ROOT
import sys

ROOT.gStyle.SetOptStat(0) # disables the stat boxes in plots

We set the seed of the data sample to be analysed, enable multithreading, and connect RDataFrame to the TTree in our created `events.root` and `events_smeared.root` files. 

Similarly to the `pythia8_generate-tree.ipynb` notebook, this notebook can be converted to a script using `jupyter nbconvert --to script plot-ea.ipynb`. In that case, the script takes a seed argument, e.g., `python3 plot-ea.py --seed 22`.

In [ ]:
# set values when running as a Jupyter notebook
if 'ipykernel' in sys.modules:
    print("Jupyter environment detected! Assuming that the analysis is ran locally as a Jupyter notebook.")

    # set the default seed
    seed = 22

    # enable multithreading, but spare one core for user operations
    nCores = max(1, os.cpu_count() - 1) 
    ROOT.EnableImplicitMT(nCores)

# otherwise, script generation is assumed and the seed is loaded from the command line
else:

    # initialize the parser which loads the seed argument
    parser = argparse.ArgumentParser()

    # define the argument to load
    parser.add_argument('--seed', type=int, required=True)

    # parse the argument from the command line
    args = parser.parse_args()

    # set the seed
    seed = args.seed

    # enable multithreading on every available core
    ROOT.EnableImplicitMT()

# connect RDataFrame to the TTree in events_smeared.root
dfSmeared = ROOT.RDataFrame("events_smeared", f"data/{seed}/events*_smeared.root")

## Definition of Computations

Afterwards, we tell RDataFrame how it should filter events and construct our observables using the `.Define()` and `.Filter()` methods. These declarations are "lazy", meaning that the event loop is not triggered until we explicitly call `.GetValue()` to extract the histograms in the next sections.

1. We define **multiplicity** as the number of tracks in the respective (true or smeared) dataset.

2. We define the calculation of the *transverse spherocity* $S_0$ and the *unweighted transverse spherocity* $S_0^{p_\mathrm{T} = 1}$ for events with $N_\mathrm{ch}(|\eta| < 1) > 10$. These observables are defined as
$$
S_0^{p_\mathrm{T} = 1} \coloneqq \frac{\pi^2}{4} \min_{\vec{n}} \left( \frac{\sum_i |\vec{u}_{\mathrm{T}, i} \times \vec{n}|}{N_\mathrm{trks}} \right)^2, \qquad 
S_0 \coloneqq \frac{\pi^2}{4} \min_{\vec{n}} \left( \frac{\sum_i |\vec{p}_{\mathrm{T}, i} \times \vec{n}|}{\sum_i |\vec{p}_{\mathrm{T}, i}|} \right)^2,
$$ 
where $\vec{u}_{\mathrm{T}, i}$ is the unit transverse momentum vector ($\vec{p}_{\mathrm{T}, i}$) of the $i$-th particle, $N_\mathrm{trks}$ is the number of particles entering the sum, and $\vec{n}$ is the unit vector which minimizes the quadratic expression.

3. $R_\mathrm{T}$ -- TO DO

In [ ]:
# we need to load the calculateS0pT1(px, py, Nch) and calculateS0(ux, uy, px, py, pT, Nch) C++ functions
ROOT.gInterpreter.ProcessLine('#include "cpp/spherocities.cpp"')

# we define a data frame which contains columns for all analysed observables 
df = (
    dfSmeared.Define("NchTrue", "pT.size()") # define the true multiplicity
             .Define("NchSmeared", "pTSmeared.size()") # define the smeared multiplicity

             # define the true branches
             .Define("px", "pT * cos(phi)")
             .Define("py", "pT * sin(phi)")
             .Define("ux", "px / pT") 
             .Define("uy", "py / pT")
             .Define("S0pT1True", "calculateS0pT1(ux, uy, NchTrue)") # define the true unweighted spherocity column
             .Define("S0True", "calculateS0(ux, uy, px, py, pT, NchTrue)") # define the true spherocity column

             # define the smeared branches
             .Define("pxSmeared", "pTSmeared * cos(phiSmeared)")
             .Define("pySmeared", "pTSmeared * sin(phiSmeared)")
             .Define("uxSmeared", "pxSmeared / pTSmeared")
             .Define("uySmeared", "pySmeared / pTSmeared")

             .Define("S0pT1Smeared", "calculateS0pT1(uxSmeared, uySmeared, NchSmeared)") # define the true unweighted spherocity column
             .Define("S0Smeared", "calculateS0(uxSmeared, uySmeared, pxSmeared, pySmeared, pTSmeared, NchSmeared)") # define the true spherocity column
             
             .Define("pTAccepted", "pT[TPCandTOFaccepted]") # create a column with true pT tracks which correspond to detected smeared pT tracks by TPC and TOF -- this is necessary for pT spectra construction
)

# we define a filtered data frame for events with defined spherocity (it does not matter if we use S0 or S0pT1 as both functions contain the same cuts)
dfSpherocity = df.Filter("S0True >= 0.0 && S0Smeared >= 0.0", "spherocity cut")

### Booking a File for Observables

We also book a snapshot which saves columns with our observables into a new `.root` file used in the `multifold-ea.ipynb` notebook by MultiFold.

In [ ]:
columnsToExtract = ["NchTrue", "NchSmeared", "S0True", "S0Smeared", "S0pT1True", "S0pT1Smeared"]

opts = ROOT.RDF.RSnapshotOptions() # ensure that observables{seed}.root file is recreated every time this notebook is ran
opts.fMode = "RECREATE"
opts.fLazy = True # ensure that .Snapshot does not trigger a histogram, but returns a smart pointer like .Histo1D
snapshotNch = df.Snapshot("observables", f"data/{seed}/observables{seed}.root", columnsToExtract, opts)

## Definition of Histograms

Next, we declare histograms of distributions, response matrices (RMs), and $p_\mathrm{T}$ spectra related histograms: 2D histograms of a given observable vs $p_\mathrm{T}$ (2D spectra) and 3D histograms which store true and smeared distributions and also response matrices for each $p_\mathrm{T}$ bin. We store the smart pointers in a dictionary.

In [ ]:
# configure histogram boundaries and number of bins
NchMax = 80
nBinsSpherocity = 50
nBinspT = 10

pointers = {}

### Multiplicity

In [ ]:
# store the pointers to the true and smeared multiplicity distributions, 2D histograms of multiplicity vs pT (0-10 GeV)
for label in ["True", "Smeared"]:
    pointers[f"Nch{label}"] = df.Histo1D((f"histo{seed}Nch{label}", label + " Multiplicity;N_{ch} [#minus];Normalized Events", NchMax, 0, NchMax), f"Nch{label}")
    pointers[f"pTNch{label}"] = df.Histo2D((f"histo{seed}pTNch{label}", label + " Multiplicity vs p_{T};p_{T}^{MC acc} [GeV/c];N_{ch} [#minus]", nBinspT, 0, 10, NchMax, 0, NchMax), "pTAccepted", f"Nch{label}")

# response matrix = 2D histogram with true Nch on the x-axis and smeared Nch on the y-axis
pointers["NchRM"] = df.Histo2D((f"histo{seed}NchRM", "Multiplicity Response Matrix;N_{ch}^{MC} [#minus];N_{ch}^{sm} [#minus]", NchMax, 0, NchMax, NchMax, 0, NchMax), "NchTrue", "NchSmeared")

# RooUnfold (which we are going to use for Bayesian unfolding) requires a response matrix with swapped axes
pointers["NchRMUnf"] = df.Histo2D((f"histo{seed}NchRMUnf", "Multiplicity Response Matrix;N_{ch}^{sm} [#minus];N_{ch}^{MC} [#minus]", NchMax, 0, NchMax, NchMax, 0, NchMax), "NchSmeared", "NchTrue")

# 3D histogram of pT, smeared, and true multiplicity
pointers["pTNchSmearedTrue"] = df.Histo3D((f"histo{seed}pTNchSmearedTrue", "Multiplicity Response Matrix vs p_{T};p_{T}^{MC acc} [GeV/c];N_{ch}^{sm} [#minus];N_{ch}^{MC} [#minus]", nBinspT, 0, 10, NchMax, 0, NchMax, NchMax, 0, NchMax), "pTAccepted", "NchSmeared", "NchTrue")

### Transverse Spherocities

In [ ]:
# define abbreviations, names, and latex labels of spherocities
abbrevs = ['S0', 'S0pT1']
names = ['Transverse Spherocity', 'Unweighted Transverse Spherocity']
latexs = ['S_{0}^{}', 'S_{0}^{p_{T} = 1}']

# store the pointers to the true and smeared distributions of spherocities, 2D histograms of spherocity vs pT (0-10 GeV)
for abbrev, name, latex in zip(abbrevs, names, latexs):
    for label in ["True", "Smeared"]:
        pointers[f"{abbrev}{label}"] = dfSpherocity.Histo1D((f"histo{seed}{abbrev}{label}", f"{label} {name};{latex} [#minus];Normalized Events", nBinsSpherocity, 0, 1), f"{abbrev}{label}")
        pointers[f"pT{abbrev}{label}"] = dfSpherocity.Histo2D((f"histo{seed}pT{abbrev}{label}", f"{label} {name} vs " + "p_{T};p_{T}^{MC acc} [GeV/c];" + latex + " [#minus]", nBinspT, 0, 10, nBinsSpherocity, 0, 1), "pTAccepted", f"{abbrev}{label}")

    # RM
    pointers[f"{abbrev}RM"] = dfSpherocity.Histo2D((f"histo{seed}{abbrev}RM", f"{name} Response Matrix;" + latex[:-1] + " MC} [#minus];" + latex[:-1] + " sm} [#minus]", nBinsSpherocity, 0, 1, nBinsSpherocity, 0, 1), f"{abbrev}True", f"{abbrev}Smeared")

    # RM for RooUnfold
    pointers[f"{abbrev}RMUnf"] = dfSpherocity.Histo2D((f"histo{seed}{abbrev}RMUnf", f"{name} Response Matrix;" + latex[:-1] + " sm} [#minus];" + latex[:-1] + " MC} [#minus]", nBinsSpherocity, 0, 1, nBinsSpherocity, 0, 1), f"{abbrev}Smeared", f"{abbrev}True")

    # 3D histogram of pT, smeared, and true spherocity
    pointers[f"pT{abbrev}SmearedTrue"] = dfSpherocity.Histo3D((f"histo{seed}pT{abbrev}SmearedTrue", name + " Response Matrix vs p_{T};p_{T}^{MC acc} [GeV/c];" + latex[:-1] + " sm} [#minus];" + latex[:-1] + " MC} [#minus]", nBinspT, 0, 10, nBinsSpherocity, 0, 1, nBinsSpherocity, 0, 1), "pTAccepted", f"{abbrev}Smeared", f"{abbrev}True")

### $R_\mathrm{T}$

TO DO

## Event Loop

Now, we trigger the event loop by calling `.GetValue()`. RDataFrame processes the tree once and fills all histograms simultaneously, while also executing the snapshots. 

In [ ]:
# fill the histograms and execute the snapshot
histos = {key: pointer.GetValue() for key, pointer in pointers.items()}
snapshotNch.GetValue()

## Plotting

In the end, we plot the distributions, RMs and 2D spectra.

We also make a file in which we will store the created histograms and response matrices for future unfolding, and a directory to store `.pdf` files in.

In [ ]:
# create the directory for images
os.makedirs(f"img/{seed}", exist_ok=True)

# create the root file for histograms
file = ROOT.TFile(f"data/{seed}/events{seed}_plots.root", "RECREATE")

### Multiplicity

We create a folder for multiplicity-related plots.

In [ ]:
os.makedirs(f"img/{seed}/Nch", exist_ok=True)

We plot the distributions of true and smeared multiplicity.

In [ ]:
# plot the distributions
for label in ["True", "Smeared"]:

    # load the histogram of the multiplicity distribution
    histoNch = histos[f"Nch{label}"]
    histoNch.Write() # store the histogram into the .root file -- note that we must save the histogram before normalizing it, as our response matrix is not constructed from the normalized distributions (and unfolding would thus fail)
    
    # set up canvas for drawing
    canvasNch = ROOT.TCanvas(f"canvas{seed}Nch{label}", f"{label} Multiplicity", 800, 600) # create a canvas to draw the histogram
    canvasNch.SetLogy() # set logarithmic scale for y-axis

    # normalize the histogram
    integral = histoNch.Integral()
    if integral > 0: # check if the histogram has entries to avoid division by zero
        histoNch.Scale(1.0 / integral) # normalize the histogram

    # customize the histogram
    histoNch.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

    histoNch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoNch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    # draw
    histoNch.Draw("HIST E") # draw the histogram

    # description
    description = ROOT.TLatex()
    description.SetNDC()
    description.SetTextFont(42)
    description.SetTextSize(0.03)

    xCoord = 0.68
    yCoord = 0.85

    description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
    description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
    description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

    description.SetTextFont(62)
    description.DrawLatex(xCoord, yCoord - 0.15, "THIS THESIS")

    # save
    canvasNch.SaveAs(f"img/{seed}/Nch/{seed}Nch{label}.pdf") # save the canvas as a PDF file

Then, we visualize the multiplicity response matrix.

In [ ]:
# load the 2D histogram
histoNchRM = histos["NchRM"] # 2D histogram with true Nch on the x-axis and smeared Nch on the y-axis

# set up canvas for drawing
canvasNchRM = ROOT.TCanvas(f"canvas{seed}NchRM", "Multiplicity Response Matrix", 800, 600)
canvasNchRM.SetLogz()

# customize the histogram
histoNchRM.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

# draw
histoNchRM.Draw("COLZ")

# description
description = ROOT.TLatex()
description.SetNDC()
description.SetTextFont(42)
description.SetTextSize(0.03)

xCoord = 0.68
yCoord = 0.85

description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

description.SetTextFont(62)
description.DrawLatex(xCoord, yCoord - 0.15, "THIS THESIS")

# save
canvasNchRM.SaveAs(f"img/{seed}/Nch/{seed}NchRM.pdf")

As RooUnfold (which we are going to use for Bayesian unfolding) requires a response matrix with swapped axes, we store such histogram in the `.root` file. 

In [ ]:
histos["NchRMUnf"].Write() # 2D histogram with smeared Nch on the x-axis and true Nch on the y-axis

### Transverse Spherocities

We create a folder for multiplicity-related plots.

In [ ]:
os.makedirs(f"img/{seed}/S0", exist_ok=True)
os.makedirs(f"img/{seed}/S0pT1", exist_ok=True)

Afterwards, we plot the distributions of $S_0^{p_\mathrm{T} = 1}$ and $S_0$.

In [ ]:
# plot the distributions
for abbrev, name, latex in zip(abbrevs, names, latexs):
    for label in ["True", "Smeared"]:
        
        # load the histogram
        histoSpherocity = histos[f"{abbrev}{label}"]
        histoSpherocity.Write() # store the histogram into the .root file

        # set up canvas for drawing
        canvasSpherocity = ROOT.TCanvas(f"canvas{seed}{abbrev}{label}", label + " " + name, 800, 600)
        canvasSpherocity.SetLogy()

        # normalize the histogram
        integral = histoSpherocity.Integral()
        if integral > 0:
            histoSpherocity.Scale(1.0 / integral)

        # customize the histogram
        histoSpherocity.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

        histoSpherocity.SetLineColor(ROOT.kRed + 1)
        histoSpherocity.SetFillColorAlpha(ROOT.kRed, 0.1)

        # draw 
        histoSpherocity.Draw("HIST E")

        # description
        description = ROOT.TLatex()
        description.SetNDC()
        description.SetTextFont(42)
        description.SetTextSize(0.03)

        description.SetTextAlign(22)
        xCoord = 0.5
        yCoord = 0.37

        description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
        description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
        description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

        description.DrawLatex(xCoord, yCoord - 0.15, "N_{ch}^{MC} > 10, N_{ch}^{sm} > 10")

        description.SetTextFont(62)
        description.DrawLatex(xCoord, yCoord - 0.20, "THIS THESIS")

        # save
        canvasSpherocity.SaveAs(f"img/{seed}/{abbrev}/{seed}{abbrev}{label}.pdf")

Finally, we plot the response matrices for both spherocities.

In [ ]:
for abbrev, name, latex in zip(abbrevs, names, latexs):
    
    # load the 2D histogram
    histoSpherocityRM = histos[f"{abbrev}RM"] # 2D histogram with true unweighted spherocity on the x-axis and smeared unweighted spherocity on the y-axis

    # set up canvas for drawing
    canvasSpherocityRM = ROOT.TCanvas(f"canvas{seed}{abbrev}RM", f"{name} Response Matrix", 800, 600)
    canvasSpherocityRM.SetLogz()

    # customize the histogram
    histoSpherocityRM.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

    # draw
    histoSpherocityRM.Draw("COLZ")

    # description
    description = ROOT.TLatex()
    description.SetNDC()
    description.SetTextFont(42)
    description.SetTextSize(0.03)

    xCoord = 0.14
    yCoord = 0.85

    description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
    description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
    description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

    description.DrawLatex(xCoord, yCoord - 0.15, "N_{ch}^{MC} > 10, N_{ch}^{sm} > 10")

    description.SetTextFont(62)
    description.DrawLatex(xCoord, yCoord - 0.20, "THIS THESIS")

    # save
    canvasSpherocityRM.SaveAs(f"img/{seed}/{abbrev}/{seed}{abbrev}RM.pdf")

Once again, we must construct and save a RooUnfold compatible response matrix. Additionally, we will save the true and smeared spherocity distribution here from the data frame with the double cut in true and smeared multiplicity: $N_\mathrm{ch}^\mathrm{MC} > 10 \quad \wedge \quad N_\mathrm{ch}^\mathrm{sm} > 10$.

In [ ]:
for abbrev, name, latex in zip(abbrevs, names, latexs):
    histos[f"{abbrev}RMUnf"].Write() # 2D histogram with true unweighted spherocity on the y-axis and smeared unweighted spherocity on the x-axis

### $R_\mathrm{T}$

TO DO

### $p_\mathrm{T}$ Spectra

In this section, we plot and save the 2D histograms of a given observable vs $p_\mathrm{T}$ (2D spectra) for unfolding. 

> **Note:** We use particle-level (true) $p_\mathrm{T}$ with simulated acceptance of TPC and TOF. If we were to unfold real experimental data, we would have to use $p_\mathrm{T}$ corrected for reconstruction efficiency (smearing) and detector acceptance (PROC KDYZ JI NAKONEC UMELE ZAVADIME -- NESTACI KOREKCE NA SMEARING?).

Mapped onto **multiplicity** classes:

In [ ]:
# dictionary for the for cycle
histospTNch = {
    "True": histos["pTNchTrue"],
    "Smeared": histos["pTNchSmeared"]
}

# plot the 2D histograms
for label, histo in histospTNch.items():
    
    # save the histogram for unfolding
    histo.Write() 

    # set up canvas
    canvaspTNch = ROOT.TCanvas(f"canvas{seed}pTNch{label}", label + " Multiplicity vs p_{T}", 800, 600)
    canvaspTNch.SetLogz()

    # customize the histogram
    histo.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

    # draw
    histo.Draw("COLZ")

    # description
    description = ROOT.TLatex()
    description.SetNDC()
    description.SetTextFont(42)
    description.SetTextSize(0.03)

    description.SetTextAlign(22)
    xCoord = 0.5
    yCoord = 0.32

    description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
    description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
    description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

    description.SetTextFont(62)
    description.DrawLatex(xCoord, yCoord - 0.15, "THIS THESIS")

    # save
    canvaspTNch.SaveAs(f"img/{seed}/Nch/{seed}pTNch{label}.pdf")

Because multiplicity distributions differ in each $p_\mathrm{T}$ bin, we will have to perform 1D unfolding of multiplicity in each bin -- **deconvoluted 1D unfolding**. Therefore, we must save a 3D histogram which stores true and smeared multiplicity distributions and also a response matrix for each $p_\mathrm{T}$ bin. A more detailed explanation can be found in the `unfolding-Bayes-ea.ipynb` notebook.

In [ ]:
# save the 3D histogram of pT, smeared, and true multiplicity
histos["pTNchSmearedTrue"].Write()

Mapped onto **spherocity** classes: 

In [ ]:
for abbrev, name, latex in zip(abbrevs, names, latexs):

    # dictionary for the for cycle
    histospTSpherocity = {
        "True": histos[f"pT{abbrev}True"],
        "Smeared": histos[f"pT{abbrev}Smeared"]
    }

    # plot the 2D histograms
    for label, histo in histospTSpherocity.items():
        
        # save the histogram for unfolding
        histo.Write() 

        # set up canvas
        canvaspTSpherocity = ROOT.TCanvas(f"canvas{seed}pT{abbrev}{label}", label + name + " vs p_{T}", 800, 600)
        canvaspTSpherocity.SetLogz()

        # customize the histogram
        histo.GetXaxis().SetTitleOffset(1.3) # increase the gap between the x-axis and its label

        # draw
        histo.Draw("COLZ")

        # description
        description = ROOT.TLatex()
        description.SetNDC()
        description.SetTextFont(42)
        description.SetTextSize(0.03)

        description.SetTextAlign(22)
        xCoord = 0.5
        yCoord = 0.37

        description.DrawLatex(xCoord, yCoord, "pp, #sqrt{s} = 510 GeV")
        description.DrawLatex(xCoord, yCoord - 0.05, "MB, PYTHIA8 (Detroit)")
        description.DrawLatex(xCoord, yCoord - 0.10, "|#eta| < 1, p_{T} > 0.2 GeV/c")

        description.DrawLatex(xCoord, yCoord - 0.15, "N_{ch}^{MC} > 10, N_{ch}^{sm} > 10")

        description.SetTextFont(62)
        description.DrawLatex(xCoord, yCoord - 0.20, "THIS THESIS")

        # save
        canvaspTSpherocity.SaveAs(f"img/{seed}/{abbrev}/{seed}pT{abbrev}{label}.pdf")

    # we also store the 3D histogram for 1D unfolding in each pT bin
    histos[f"pT{abbrev}SmearedTrue"].Write()

In the end, we close the `.root` file.

In [ ]:
file.Close()